Evaluating ASR

In [1]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec spacy
!python -m spacy download en_core_web_trf

/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 52.0 MB/s  0:00:08:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


## Setup

In [2]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Test Data Setup

In [3]:
import torchaudio
from IPython.display import Audio, display




dataset_paths = [("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test", "voxpopuli"),
                 ("/training-1/asr/kaldi_data/cv19/en/test", "commonvoice"),
                 #("/training-1/asr/kaldi_data/singleword/en/test", "commonvoice_singlewords"),
                 ("/training-1/asr/kaldi_data/mls/en/test", "librispeech")

                 
                ]# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es
datasets = {}
for dataset_path, dataset_title in dataset_paths:
    dataset = []
    wav_scp_dict = {}
    with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
        for line in rf:
            id, wav_path = line.strip().split("\t")
            wav_scp_dict[id] = wav_path
    
    text_dict = {}
    with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
        for line in rf:
            id, text = line.strip().split("\t")
            text_dict[id] = text
    
    for id, wav_path in wav_scp_dict.items():
        text = text_dict[id]
        dataset.append({"audio" : wav_path, "text" : text})
    datasets[dataset_title] = dataset

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
'''
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))
'''


'\nfirst_entry = dataset[0]\nprint(first_entry)\nprint("-"*10)\nprint(first_entry["text"])\naudio = load_audio(first_entry["audio"])\ndisplay(Audio(audio, rate=16000))\n'

## Transcription Function

In [4]:
def transcribe(audio_path, prompt=None, language=None):
    kwargs = {"audio": audio_path}
    if prompt:
        kwargs["prompt"] = prompt
    if language:
        kwargs["language"] = language
    inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)

    output_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    return processor.decode(generated_ids, return_format="transcription_only")[0].strip()

## Text Normalization, Imports

In [5]:
import jiwer
import re
import string

def normalize(text):
    text = text.translate(str.maketrans('', '', string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

## Run Inference on the Test Set

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_trf")
context_list = [False, True]

for c in context_list:
    for dataset_title, dataset_entries in datasets.items():
        eval_rows = []
        for sample in tqdm(dataset_entries):
            entities = None
            if c:
                doc = nlp(sample["text"])
                entities = [ent.text for ent in doc.ents]
                entities = ", ".join(entities)
            model_output = None
            if entities:
                model_output = transcribe(sample["audio"], prompt=entities)
            else:
                model_output = transcribe(sample["audio"])
            wer = jiwer.wer(sample["text"], model_output)
            cer = jiwer.cer(sample["text"], model_output)
            eval_rows.append({"audio_path": sample["audio"],
                              "label": normalize(sample["text"]),
                              "prediction": normalize(model_output),
                              "wer" : wer,
                              "cer": cer
                             }
                         )
        output_statistics = []
        results = pd.DataFrame(eval_rows)
        results.head()
        output_statistics.append(f"Mean WER: {results["wer"].mean()}")
        output_statistics.append(f"Mean WER: {results["cer"].mean()}")
        results = results.sort_values("wer", ascending=False)
        output_name = ""
        if c:
            output_name = f"qwen_{dataset_title}_with_context"
        else:
            output_name = f"qwen_{dataset_title}_without_context"
        results.to_csv(f"{output_name}.tsv", sep="\t", index=False)
        with open(f"{output_name}_statistics.txt", "w", encoding="UTF-8") as wf:
            for line in output_statistics:
                wf.write(line+"\n")
        print("="*50)
        print(dataset_title)
        print(output_name)
        print("\n".join(output_statistics))
        print("="*50)

/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


  0%|          | 0/1842 [00:00<?, ?it/s]

voxpopuli
qwen_voxpopuli_without_context
Mean WER: 0.29505862025268426
Mean WER: 0.09371622362188697


  0%|          | 0/16396 [00:00<?, ?it/s]

## Corpus-Level Results

In [ ]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
results[["wer", "cer"]].describe()
results = results.sort_values("wer", ascending=False)
results.to_csv("results_with_context.tsv", sep="\t", index=False)

## Worst Samples

In [ ]:
results.sort_values("wer", ascending=False).head(10)


In [ ]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio_path"])
    print("LABEL    :", row["label"])
    print("PRED    :", row["prediction"])
    print(f"WER {row["wer"]:.2%} | CER {row["cer"]:.2%}")
    display(Audio(load_audio(row["audio_path"]), rate=16000))
